# FaceGuard — Training Notebook
## Cours IA — TP3, TP4, TP5, TP6, TP9
**Université Marie et Louis Pasteur — Prof. Christophe Guyeux**

Ce notebook entraîne tous les modèles XGBoost sur le dataset 18K visages,
génère les analyses SHAP, et évalue les performances.

### Plan
1. Chargement et exploration du dataset (TP1 — pandas)
2. Extraction des features (HOG, LBP, FFT, FaceNet)
3. Clustering des embeddings (TP2 — k-means, t-SNE)
4. Classification deepfake (TP3, TP4 — XGBoost)
5. Régression densité foule (TP5)
6. Analyse temporelle comportementale (TP6)
7. Explicabilité SHAP (TP9)

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
print('Imports OK')

ModuleNotFoundError: No module named 'numpy'

## 1. Chargement et exploration du dataset (TP1 — Pandas)

In [ ]:
import os
from PIL import Image

REAL_DIR = Path('../data/faces/real')
FAKE_DIR = Path('../data/faces/fake')

real_paths = list(REAL_DIR.glob('*.jpg')) + list(REAL_DIR.glob('*.png'))
fake_paths = list(FAKE_DIR.glob('*.jpg')) + list(FAKE_DIR.glob('*.png'))

print(f'Real faces : {len(real_paths):,}')
print(f'Fake faces : {len(fake_paths):,}')
print(f'Total      : {len(real_paths)+len(fake_paths):,}')

# Build metadata DataFrame (TP1)
rows = []
for p in real_paths[:5000]:
    rows.append({'path': str(p), 'label': 0, 'label_name': 'real',
                 'size_kb': p.stat().st_size / 1024})
for p in fake_paths[:5000]:
    rows.append({'path': str(p), 'label': 1, 'label_name': 'fake',
                 'size_kb': p.stat().st_size / 1024})

df = pd.DataFrame(rows)
print('\nDataset stats:')
print(df.groupby('label_name')['size_kb'].describe().round(2))

In [ ]:
# Visualise sample images
import cv2

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, p in enumerate(real_paths[:5]):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title('REAL', color='green', fontsize=10)
    axes[0, i].axis('off')
for i, p in enumerate(fake_paths[:5]):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img)
    axes[1, i].set_title('FAKE (GAN)', color='red', fontsize=10)
    axes[1, i].axis('off')
plt.suptitle('Dataset samples — Real vs GAN-generated faces', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Extraction des features

In [ ]:
# Load pre-extracted features if available, else extract
REAL_NPY = Path('../data/features_real.npy')
FAKE_NPY = Path('../data/features_fake.npy')

if REAL_NPY.exists() and FAKE_NPY.exists():
    X_real = np.load(REAL_NPY)
    X_fake = np.load(FAKE_NPY)
    print(f'Loaded pre-extracted features: real={X_real.shape}, fake={X_fake.shape}')
else:
    print('Running feature extraction (this may take ~30 minutes on first run)...')
    print('Run train_models.py first, then reload this cell.')
    # Fallback: generate random features for notebook demo
    X_real = np.random.randn(500, 512).astype(np.float32)
    X_fake = np.random.randn(500, 512).astype(np.float32) + 0.3
    print('Using synthetic features for demo.')

X = np.vstack([X_real, X_fake])
y = np.array([0]*len(X_real) + [1]*len(X_fake))
print(f'Full dataset: X={X.shape}, y={y.shape}')

## 3. Clustering des embeddings (TP2 — K-means + t-SNE/UMAP)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Subsample for visualisation
idx = np.random.choice(len(X), min(1000, len(X)), replace=False)
X_sub, y_sub = X[idx], y[idx]

# PCA first for speed
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_sub)
print(f'PCA variance explained (50 components): {pca.explained_variance_ratio_.sum():.1%}')

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_pca)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#22c55e' if label == 0 else '#dc2626' for label in y_sub]
ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors, alpha=0.6, s=20)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#22c55e', label='Real'), Patch(color='#dc2626', label='Fake (GAN)')])
ax.set_title('t-SNE — Embedding space: real vs GAN-generated faces (TP2bis)', fontsize=13)
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.savefig('../docs/tsne_embeddings.png', dpi=150)
plt.show()
print('t-SNE plot saved to docs/tsne_embeddings.png')

## 4. Classification Deepfake — XGBoost (TP3, TP4)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    RocCurveDisplay, ConfusionMatrixDisplay
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

# XGBoost model (TP4)
model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    gamma=0.1,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100,
)
print('Training complete.')

In [ ]:
# Evaluation (TP3)
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Real','Fake'],
    colorbar=False, ax=axes[0], cmap='Blues'
)
axes[0].set_title('Confusion Matrix')
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title('ROC Curve — Deepfake Detector')
axes[1].plot([0,1],[0,1],'--',color='gray')
plt.tight_layout()
plt.savefig('../docs/evaluation_deepfake.png', dpi=150)
plt.show()

## 5. Explicabilité SHAP (TP9)

In [ ]:
import shap

# Feature names
feature_names = (
    [f'embed_{i}'  for i in range(128)] +
    [f'hog_{i}'    for i in range(128)] +
    [f'lbp_{i}'    for i in range(256)]
)

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test[:300])

print('=== SHAP Summary Plot (TP9) ===')
shap.summary_plot(
    shap_values, X_test[:300],
    feature_names=feature_names,
    max_display=20,
    show=False
)
plt.title('SHAP Feature Importance — Deepfake classifier')
plt.tight_layout()
plt.savefig('../docs/shap_summary.png', dpi=150)
plt.show()

# Feature group importance
shap_abs = np.abs(shap_values)
groups = {
    'FaceNet embedding (0-127)' : shap_abs[:, :128].mean(),
    'HOG texture (128-255)'     : shap_abs[:, 128:256].mean(),
    'LBP histogram (256-511)'   : shap_abs[:, 256:].mean(),
}
print('\nMean |SHAP| by feature group:')
for k, v in sorted(groups.items(), key=lambda x: -x[1]):
    print(f'  {k:35s}: {v:.5f}')

In [ ]:
# Waterfall plot for one prediction
idx_fake = np.where(y_test == 1)[0][0]
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx_fake],
        base_values=explainer.expected_value,
        data=X_test[idx_fake],
        feature_names=feature_names
    ),
    max_display=15,
    show=False
)
plt.title('SHAP Waterfall — Single fake face explanation')
plt.tight_layout()
plt.savefig('../docs/shap_waterfall.png', dpi=150)
plt.show()

## 6. Régression densité de foule (TP5)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Synthetic crowd dataset (replace with real crowd images in production)
np.random.seed(42)
n = 2000
face_count  = np.random.randint(0, 50, n)
brightness  = np.random.uniform(80, 200, n)
motion      = np.random.uniform(0, 1, n)
area_m2     = np.random.choice([150, 800, 1200, 2000], n)
density     = face_count / area_m2 + np.random.normal(0, 0.001, n)
density     = np.clip(density, 0, 5)

X_crowd = np.column_stack([face_count, brightness, motion, area_m2])
y_crowd = density

X_ct, X_cv, y_ct, y_cv = train_test_split(X_crowd, y_crowd, test_size=0.2, random_state=42)

crowd_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    objective='reg:squarederror', random_state=42
)
crowd_model.fit(X_ct, y_ct, eval_set=[(X_cv, y_cv)], verbose=50)

y_cv_pred = crowd_model.predict(X_cv)
print(f'MAE  : {mean_absolute_error(y_cv, y_cv_pred):.5f} persons/m²')
print(f'RMSE : {mean_squared_error(y_cv, y_cv_pred)**0.5:.5f}')
print(f'R²   : {r2_score(y_cv, y_cv_pred):.4f}')

plt.figure(figsize=(8, 5))
plt.scatter(y_cv, y_cv_pred, alpha=0.3, s=10)
plt.plot([0, y_cv.max()], [0, y_cv.max()], 'r--')
plt.xlabel('Densité réelle (persons/m²)')
plt.ylabel('Densité prédite')
plt.title('Régression densité foule — XGBoost (TP5)')
plt.tight_layout()
plt.savefig('../docs/crowd_regression.png', dpi=150)
plt.show()

## 7. Analyse comportementale — Séries temporelles (TP6)

In [ ]:
# Simulate position time series for 3 behavior types
t = np.linspace(0, 60, 300)  # 60 seconds, 5 FPS

# Normal movement
pos_normal = np.column_stack([
    0.5 + 0.05*np.sin(t*0.3) + np.random.randn(300)*0.01,
    0.5 + 0.05*np.cos(t*0.3) + np.random.randn(300)*0.01
])

# Loitering (almost stationary)
pos_loiter = np.column_stack([
    0.4 + np.random.randn(300)*0.005,
    0.6 + np.random.randn(300)*0.005
])

# Running (fast linear motion)
pos_run = np.column_stack([
    np.linspace(0.1, 0.9, 300) + np.random.randn(300)*0.01,
    0.5 + np.random.randn(300)*0.01
])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, pos, title, color in zip(
    axes,
    [pos_normal, pos_loiter, pos_run],
    ['Normal (TP6 — série temporelle)', 'Loitering — Suspect', 'Running — Alerte'],
    ['#22c55e', '#f97316', '#dc2626']
):
    ax.plot(pos[:, 0], pos[:, 1], color=color, alpha=0.7, linewidth=1.5)
    ax.scatter(pos[0, 0], pos[0, 1], color='black', s=60, zorder=5, label='Start')
    ax.scatter(pos[-1, 0], pos[-1, 1], color=color, s=80, marker='*', zorder=5, label='End')
    ax.set_title(title, fontsize=11)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Position X (normalisée)')
    ax.set_ylabel('Position Y (normalisée)')
    ax.legend(fontsize=8)
plt.suptitle('Trajectoires comportementales — Agent 3 (TP6 séries temporelles)', fontsize=13)
plt.tight_layout()
plt.savefig('../docs/behavior_trajectories.png', dpi=150)
plt.show()

## 8. Sauvegarde des modèles

In [ ]:
import pickle, os
os.makedirs('../models', exist_ok=True)

with open('../models/xgboost_deepfake.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('../models/xgboost_crowd.pkl', 'wb') as f:
    pickle.dump(crowd_model, f)

print('Models saved:')
for p in Path('../models').glob('*.pkl'):
    print(f'  {p.name:35s} {p.stat().st_size/1024:.1f} KB')

## Résumé — couverture cours_ia

| TP | Contenu | Cellule |
|----|---------|--------|
| TP1 | Pandas, exploration dataset | Cellule 2 |
| TP2 | K-means, clustering | Cellule 4 |
| TP2bis | t-SNE, UMAP | Cellule 4 |
| TP3 | Classification supervisée | Cellule 5–6 |
| TP4 | XGBoost avancé | Cellule 5 |
| TP5 | Régression | Cellule 8 |
| TP6 | Séries temporelles | Cellule 9 |
| TP9 | SHAP, explicabilité | Cellule 7 |